In [1]:
import pandas as pd
import lxml
import requests
import yfinance as yf
from io import StringIO

In [2]:
url= "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers= {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [3]:
response=requests.get(url, headers=headers)

sp500=pd.read_html(StringIO(response.text))[0]
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [4]:
sp500['Symbol']=sp500['Symbol'].str.replace(".", "-", regex=False)

In [5]:
# list of ticker symbols for download (Needs to be done for S&P 500)
tickerStrings = sp500['Symbol'].tolist()

In [6]:
df = yf.download(tickerStrings, group_by='Ticker', period='1d')

[*********             18%                       ]  93 of 503 completed$AMCR: possibly delisted; no price data found  (period=1d)
[**********            20%                       ]  101 of 503 completed$HD: possibly delisted; no price data found  (period=1d)
[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['AMCR', 'HD']: possibly delisted; no price data found  (period=1d)


In [7]:
df

Ticker            INCY                                                 FTV  \
Price             Open        High         Low   Close   Volume       Open   
Date                                                                         
2026-06-26  109.480003  115.559998  109.480003  113.75  3794900  61.630001   

Ticker                                        ...    CDNS              \
Price        High        Low  Close   Volume  ...    Open        High   
Date                                          ...                       
2026-06-26  61.73  61.009998  61.48  3560300  ...  362.25  381.899994   

Ticker                                             GM                   \
Price             Low       Close   Volume       Open  High        Low   
Date                                                                     
2026-06-26  358.76001  377.269989  3904700  78.389999  79.5  77.809998   

Ticker                           
Price           Close    Volume  
Date                             
2026-06-26  78.099998  12591100  

[1 rows x 2517 columns]

In [8]:
df = df.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
df

Price,Date,Ticker,Open,High,Low,Close,Volume,Adj Close
0,2026-06-26,INCY,109.480003,115.559998,109.480003,113.750000,3794900.0,NaN
1,2026-06-26,FTV,61.630001,61.730000,61.009998,61.480000,3560300.0,NaN
2,2026-06-26,NXPI,289.170013,291.339996,275.179993,277.019989,7172900.0,NaN
3,2026-06-26,NVDA,193.119995,195.550003,191.220001,192.529999,178906300.0,NaN
4,2026-06-26,XYL,117.360001,117.800003,115.750000,116.449997,2606000.0,NaN
...,...,...,...,...,...,...,...,...
498,2026-06-26,APO,121.480003,122.089996,117.570000,118.290001,11693800.0,NaN
499,2026-06-26,BSX,44.400002,45.220001,44.020000,44.230000,30003800.0,NaN
500,2026-06-26,AXP,341.989990,344.010010,338.320007,340.359985,5899100.0,NaN
501,2026-06-26,CDNS,362.250000,381.899994,358.760010,377.269989,3904700.0,NaN


In [9]:
downloaded_tickers=set(df["Ticker"].unique())

In [10]:
missing = set(tickerStrings) - downloaded_tickers

missing

set()

In [11]:
len(downloaded_tickers)

503

In [12]:
row_counts = df.groupby("Ticker").size()
print(row_counts.sort_values())

Ticker
A       1
AAPL    1
ABBV    1
ABNB    1
ABT     1
       ..
XYZ     1
YUM     1
ZBH     1
ZBRA    1
ZTS     1
Length: 503, dtype: int64


In [13]:
for ticker in ["FRT", "PHM", "SYY"]:
    count = (df["Ticker"] == ticker).sum()
    print(f"{ticker}: {count} rows")

FRT: 1 rows
PHM: 1 rows
SYY: 1 rows


In [14]:
df.columns.name = None
df

,Date,Ticker,Open,High,Low,Close,Volume,Adj Close
0,2026-06-26,INCY,109.480003,115.559998,109.480003,113.750000,3794900.0,NaN
1,2026-06-26,FTV,61.630001,61.730000,61.009998,61.480000,3560300.0,NaN
2,2026-06-26,NXPI,289.170013,291.339996,275.179993,277.019989,7172900.0,NaN
3,2026-06-26,NVDA,193.119995,195.550003,191.220001,192.529999,178906300.0,NaN
4,2026-06-26,XYL,117.360001,117.800003,115.750000,116.449997,2606000.0,NaN
...,...,...,...,...,...,...,...,...
498,2026-06-26,APO,121.480003,122.089996,117.570000,118.290001,11693800.0,NaN
499,2026-06-26,BSX,44.400002,45.220001,44.020000,44.230000,30003800.0,NaN
500,2026-06-26,AXP,341.989990,344.010010,338.320007,340.359985,5899100.0,NaN
501,2026-06-26,CDNS,362.250000,381.899994,358.760010,377.269989,3904700.0,NaN


In [15]:
ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]

In [18]:
downloaded_tickers = set(df["Ticker"].unique())

downloaded_tickers

{'A',
 'AAPL',
 'ABBV',
 'ABNB',
 'ABT',
 'ACGL',
 'ACN',
 'ADBE',
 'ADI',
 'ADM',
 'ADP',
 'ADSK',
 'AEE',
 'AEP',
 'AES',
 'AFL',
 'AIG',
 'AIZ',
 'AJG',
 'AKAM',
 'ALB',
 'ALGN',
 'ALL',
 'ALLE',
 'AMAT',
 'AMCR',
 'AMD',
 'AME',
 'AMGN',
 'AMP',
 'AMT',
 'AMZN',
 'ANET',
 'AON',
 'AOS',
 'APA',
 'APD',
 'APH',
 'APO',
 'APP',
 'APTV',
 'ARE',
 'ARES',
 'ATO',
 'AVB',
 'AVGO',
 'AVY',
 'AWK',
 'AXON',
 'AXP',
 'AZO',
 'BA',
 'BAC',
 'BALL',
 'BAX',
 'BBY',
 'BDX',
 'BEN',
 'BF-B',
 'BG',
 'BIIB',
 'BKNG',
 'BKR',
 'BLDR',
 'BLK',
 'BMY',
 'BNY',
 'BR',
 'BRK-B',
 'BRO',
 'BSX',
 'BX',
 'BXP',
 'C',
 'CAG',
 'CAH',
 'CARR',
 'CASY',
 'CAT',
 'CB',
 'CBOE',
 'CBRE',
 'CCI',
 'CCL',
 'CDNS',
 'CDW',
 'CEG',
 'CF',
 'CFG',
 'CHD',
 'CHRW',
 'CHTR',
 'CI',
 'CIEN',
 'CINF',
 'CL',
 'CLX',
 'CMCSA',
 'CME',
 'CMG',
 'CMI',
 'CMS',
 'CNC',
 'CNP',
 'COF',
 'COHR',
 'COIN',
 'COO',
 'COP',
 'COR',
 'COST',
 'CPAY',
 'CPRT',
 'CPT',
 'CRH',
 'CRL',
 'CRM',
 'CRWD',
 'CSCO',
 'CSGP',
 'CSX',


In [19]:
fully_missing = set(tickerStrings) - downloaded_tickers

fully_missing

set()

In [22]:
def validate_download(df, tickerStrings, ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]):
    
    downloaded_tickers = set(df["Ticker"].unique())
    fully_missing = set(tickerStrings) - downloaded_tickers

    # Per-column null counts per ticker
    null_by_col = (
        df.groupby("Ticker")[ohlcv_cols]
        .apply(lambda x: x.isna().sum())
    )
    null_by_col["Total"] = null_by_col.sum(axis=1)
    has_nulls = null_by_col[null_by_col["Total"] > 0]

    # First and last valid (non-null) date per ticker
    valid_dates = (
        df[df["Close"].notna()]
        .groupby("Ticker")["Date"]
        .agg(
            first_valid_date="min",
            last_valid_date="max"
        )
    )

    # Get the earliest date in your entire dataset (start of your backfill window)
    backfill_start = df["Date"].min()
    
    # Count trading days between backfill start and first valid date per ticker
    # Trading days = number of dates that appear in your df for any ticker
    all_trading_days = df["Date"].drop_duplicates().sort_values()
    
    def count_trading_days_before(first_valid_date):
        return (all_trading_days < first_valid_date).sum()
    
    valid_dates["trading_days_before_listing"] = valid_dates["first_valid_date"].apply(
        count_trading_days_before
    )
    # Distinguish fully null vs partially null
    row_counts = df.groupby("Ticker").size()
    max_possible_nulls = row_counts * len(ohlcv_cols)
    fully_null_mask = has_nulls["Total"] == max_possible_nulls[has_nulls.index]
    fully_null = has_nulls[fully_null_mask]
    partially_null = has_nulls[~fully_null_mask].sort_values("Total", ascending=False)

    # Join valid date info onto partially null summary
    partially_null_summary = partially_null.join(valid_dates, how="left")

    # Report
    print(f"Total tickers expected:  {len(tickerStrings)}")
    print(f"Total tickers downloaded: {len(downloaded_tickers)}")

    if fully_missing:
        print(f"\nFully missing (not in df at all): {fully_missing}")
    else:
        print("\nFully missing: none")

    if len(fully_null) > 0:
        print(f"\nPresent but completely null (failed download):")
        print(fully_null.to_string())
    else:
        print("\nCompletely null tickers: none")

    if len(partially_null_summary) > 0:
        print(f"\nPartially null (recent IPO/spin-off) — breakdown by column + valid date range:\n")
        print(partially_null_summary.to_string())
    else:
        print("\nPartially null tickers: none")

    return fully_missing, fully_null, partially_null_summary

In [23]:
fully_missing, fully_null, partially_null = validate_download(df, tickerStrings)

Total tickers expected:  503
Total tickers downloaded: 503

Fully missing: none

Present but completely null (failed download):
        Open  High  Low  Close  Volume  Total
Ticker                                       
AMCR       1     1    1      1       1      5
HD         1     1    1      1       1      5

Partially null tickers: none


In [ ]:
df.to_csv('daily_data_2306.csv')

In [24]:
ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]

In [25]:
null_counts_by_ticker = (
    df.groupby("Ticker")[ohlcv_cols]
    .apply(lambda x: x.isna().sum().sum())  # total nulls across all OHLCV cols, per ticker
    .sort_values(ascending=False)
)

print(null_counts_by_ticker.head(10))

Ticker
HD      5
AMCR    5
NWSA    0
NWS     0
NVR     0
NVDA    0
NUE     0
NTRS    0
NTAP    0
NSC     0
dtype: int64


In [26]:
yf.download("GOOG", group_by='Ticker', period='1d')

[*********************100%***********************]  1 of 1 completed


Ticker            GOOG                                              
Price             Open        High         Low       Close    Volume
Date                                                                
2026-06-26  340.980011  344.119995  333.690002  334.690002  82450400